In [1]:
from google.colab import files
uploaded = files.upload()

Saving IPO Performance Tracker.csv to IPO Performance Tracker.csv


In [2]:
import pandas as pd

df = pd.read_csv('IPO Performance Tracker.csv')
df.head()

,S.No,Company Name,NSE Symbol,Sector,Issue Price (₹),Issue Size (Cr),Listing Date,Listing Day Close (₹),Listing Gain %,Current Price (₹),Current Gain %,Performance Category
0,1,Bharti Hexacom,BHARTIHEXA,Telecommunication,570,4275.0,16-04-2024,813.30,42.68,1477.50,159.21,Blockbuster
1,2,Swiggy,SWIGGY,Services,390,11327.0,13-11-2024,456.00,16.92,241.95,-37.96,Disaster
2,3,Hyundai Motor India,HYUNDAI,Automobile,1960,27870.0,22-10-2024,1819.60,-7.16,1945.70,-0.73,Weak
3,4,NTPC Green Energy,NTPCGREEN,Power,108,10000.0,27-11-2024,121.65,12.64,97.56,-9.67,Weak
4,5,Vishal Mega Mart,VMM,Services,78,8000.0,18-12-2024,111.93,43.50,118.14,51.46,Blockbuster


In [3]:
import sqlite3

conn = sqlite3.connect('ipo_tracker.db')
cursor = conn.cursor()
df.columns = ['sno', 'company_name', 'nse_symbol', 'sector', 'issue_price',
               'issue_size_cr', 'listing_date', 'listing_close',
               'listing_gain_pct', 'current_price', 'current_gain_pct',
               'performance_category']

df.head()

,sno,company_name,nse_symbol,sector,issue_price,issue_size_cr,listing_date,listing_close,listing_gain_pct,current_price,current_gain_pct,performance_category
0,1,Bharti Hexacom,BHARTIHEXA,Telecommunication,570,4275.0,16-04-2024,813.30,42.68,1477.50,159.21,Blockbuster
1,2,Swiggy,SWIGGY,Services,390,11327.0,13-11-2024,456.00,16.92,241.95,-37.96,Disaster
2,3,Hyundai Motor India,HYUNDAI,Automobile,1960,27870.0,22-10-2024,1819.60,-7.16,1945.70,-0.73,Weak
3,4,NTPC Green Energy,NTPCGREEN,Power,108,10000.0,27-11-2024,121.65,12.64,97.56,-9.67,Weak
4,5,Vishal Mega Mart,VMM,Services,78,8000.0,18-12-2024,111.93,43.50,118.14,51.46,Blockbuster


In [4]:
cursor.execute('''
CREATE TABLE IF NOT EXISTS ipo_master (
    ipo_id INTEGER PRIMARY KEY,
    company_name TEXT,
    nse_symbol TEXT,
    sector TEXT,
    issue_price REAL,
    issue_size_cr REAL,
    listing_date TEXT
)
''')

cursor.execute('''
CREATE TABLE IF NOT EXISTS ipo_performance (
    ipo_id INTEGER,
    listing_close REAL,
    listing_gain_pct REAL,
    current_price REAL,
    current_gain_pct REAL,
    performance_category TEXT,
    FOREIGN KEY (ipo_id) REFERENCES ipo_master(ipo_id)
)
''')

conn.commit()

In [5]:
# Table 1: ipo_master
ipo_master_df = df[['sno', 'company_name', 'nse_symbol', 'sector',
                      'issue_price', 'issue_size_cr', 'listing_date']].copy()
ipo_master_df.columns = ['ipo_id', 'company_name', 'nse_symbol', 'sector',
                           'issue_price', 'issue_size_cr', 'listing_date']

ipo_master_df.to_sql('ipo_master', conn, if_exists='replace', index=False)

# Table 2: ipo_performance
ipo_performance_df = df[['sno', 'listing_close', 'listing_gain_pct',
                           'current_price', 'current_gain_pct',
                           'performance_category']].copy()
ipo_performance_df.columns = ['ipo_id', 'listing_close', 'listing_gain_pct',
                                'current_price', 'current_gain_pct',
                                'performance_category']

ipo_performance_df.to_sql('ipo_performance', conn, if_exists='replace', index=False)

conn.commit()
print("Data loaded successfully!")

Data loaded successfully!


In [6]:
# Check both tables
print("IPO Master:")
print(pd.read_sql("SELECT * FROM ipo_master LIMIT 5", conn))

print("\nIPO Performance:")
print(pd.read_sql("SELECT * FROM ipo_performance LIMIT 5", conn))

IPO Master:
   ipo_id         company_name  nse_symbol             sector  issue_price  \
0       1       Bharti Hexacom  BHARTIHEXA  Telecommunication          570   
1       2               Swiggy      SWIGGY           Services          390   
2       3  Hyundai Motor India     HYUNDAI         Automobile         1960   
3       4    NTPC Green Energy   NTPCGREEN              Power          108   
4       5     Vishal Mega Mart         VMM           Services           78   

   issue_size_cr listing_date  
0         4275.0   16-04-2024  
1        11327.0   13-11-2024  
2        27870.0   22-10-2024  
3        10000.0   27-11-2024  
4         8000.0   18-12-2024  

IPO Performance:
   ipo_id  listing_close  listing_gain_pct  current_price  current_gain_pct  \
0       1         813.30             42.68        1477.50            159.21   
1       2         456.00             16.92         241.95            -37.96   
2       3        1819.60             -7.16        1945.70             -0

In [7]:
query1 = '''
SELECT m.company_name, m.sector, p.current_gain_pct,
CASE
    WHEN p.current_gain_pct > 50 THEN 'Blockbuster'
    WHEN p.current_gain_pct >= 20 THEN 'Strong'
    WHEN p.current_gain_pct >= 0 THEN 'Moderate'
    WHEN p.current_gain_pct >= -20 THEN 'Weak'
    ELSE 'Disaster'
END AS category
FROM ipo_master m
JOIN ipo_performance p ON m.ipo_id = p.ipo_id
ORDER BY p.current_gain_pct DESC
'''

result1 = pd.read_sql(query1, conn)
print(result1)

                    company_name                  sector  current_gain_pct  \
0                 Bharti Hexacom       Telecommunication            159.21   
1               Premier Energies                   Power            137.42   
2            Diffusion Engineers         Metals & Mining            111.31   
3                Waaree Energies                   Power             99.50   
4                 BLS E-Services  Information Technology             66.99   
5               Vishal Mega Mart                Services             51.46   
6         Aadhar Housing Finance      Financial Services             44.73   
7                 Sagility India  Information Technology             32.17   
8              Vodafone Idea FPO       Telecommunication             30.45   
9             Sambhv Steel Tubes         Metals & Mining             26.17   
10           ACME Solar Holdings                   Power             16.61   
11                     JNK India            Construction        

In [8]:
query2 = '''
SELECT m.company_name, m.sector, p.current_gain_pct,
RANK() OVER (PARTITION BY m.sector ORDER BY p.current_gain_pct DESC) as sector_rank,
ROUND(AVG(p.current_gain_pct) OVER (PARTITION BY m.sector), 2) as sector_avg_gain
FROM ipo_master m
JOIN ipo_performance p ON m.ipo_id = p.ipo_id
ORDER BY m.sector, sector_rank
'''

result2 = pd.read_sql(query2, conn)
print(result2)


                    company_name                  sector  current_gain_pct  \
0            Hyundai Motor India              Automobile             -0.73   
1                  Kross Limited              Automobile            -26.01   
2                   Ola Electric              Automobile            -41.17   
3                      JNK India            Construction             14.55   
4                  Ceigall India            Construction            -12.08   
5          Afcons Infrastructure            Construction            -31.36   
6           Globe Civil Projects            Construction            -46.14   
7            Arisinfra Solutions            Construction            -49.39   
8         Aadhar Housing Finance      Financial Services             44.73   
9     Niva Bupa Health Insurance      Financial Services             12.97   
10    Go Digit General Insurance      Financial Services              9.04   
11        HDB Financial Services      Financial Services        

In [9]:
query3 = '''
WITH sector_avg AS (
    SELECT m.sector, AVG(p.current_gain_pct) as avg_gain
    FROM ipo_master m
    JOIN ipo_performance p ON m.ipo_id = p.ipo_id
    GROUP BY m.sector
)
SELECT m.company_name, m.sector, p.current_gain_pct,
       ROUND(s.avg_gain, 2) as sector_avg,
       ROUND(p.current_gain_pct - s.avg_gain, 2) as alpha
FROM ipo_master m
JOIN ipo_performance p ON m.ipo_id = p.ipo_id
JOIN sector_avg s ON m.sector = s.sector
WHERE p.current_gain_pct > s.avg_gain
ORDER BY alpha DESC
'''

result3 = pd.read_sql(query3, conn)
print(result3)

                  company_name                  sector  current_gain_pct  \
0             Premier Energies                   Power            137.42   
1          Diffusion Engineers         Metals & Mining            111.31   
2             Vishal Mega Mart                Services             51.46   
3               Bharti Hexacom       Telecommunication            159.21   
4                    JNK India            Construction             14.55   
5              Waaree Energies                   Power             99.50   
6       Aadhar Housing Finance      Financial Services             44.73   
7          Hyundai Motor India              Automobile             -0.73   
8               BLS E-Services  Information Technology             66.99   
9                Ceigall India            Construction            -12.08   
10  Niva Bupa Health Insurance      Financial Services             12.97   

    sector_avg  alpha  
0        60.97  76.46  
1        35.58  75.73  
2       -22.85 

In [10]:
query4 = '''
WITH outperformers AS (
    SELECT m.ipo_id
    FROM ipo_master m
    JOIN ipo_performance p ON m.ipo_id = p.ipo_id
    JOIN (
        SELECT sector, AVG(current_gain_pct) as avg_gain
        FROM ipo_master m2
        JOIN ipo_performance p2 ON m2.ipo_id = p2.ipo_id
        GROUP BY sector
    ) s ON m.sector = s.sector
    WHERE p.current_gain_pct > s.avg_gain
)
SELECT m.company_name, m.sector, p.current_gain_pct, p.performance_category
FROM ipo_master m
JOIN ipo_performance p ON m.ipo_id = p.ipo_id
LEFT JOIN outperformers o ON m.ipo_id = o.ipo_id
WHERE o.ipo_id IS NULL
ORDER BY p.current_gain_pct ASC
'''

result4 = pd.read_sql(query4, conn)
print(result4)


                    company_name                  sector  current_gain_pct  \
0           Brainbees (FirstCry)                Services            -54.26   
1            Arisinfra Solutions            Construction            -49.39   
2               Western Carriers                Services            -47.17   
3           Globe Civil Projects            Construction            -46.14   
4                   Ola Electric              Automobile            -41.17   
5                         Swiggy                Services            -37.96   
6          Afcons Infrastructure            Construction            -31.36   
7          Awfis Space Solutions                Services            -26.32   
8                  Kross Limited              Automobile            -26.01   
9         HDB Financial Services      Financial Services            -15.79   
10             NTPC Green Energy                   Power             -9.67   
11  Ellenbarrie Industrial Gases         Metals & Mining        

In [11]:
query5 = '''
SELECT m.company_name, m.sector,
       p.listing_gain_pct,
       p.current_gain_pct,
       ROUND(p.current_gain_pct - p.listing_gain_pct, 2) as change_since_listing,
       CASE
           WHEN p.current_gain_pct > p.listing_gain_pct THEN 'Gained Further'
           WHEN p.current_gain_pct < p.listing_gain_pct THEN 'Cooled Down'
           ELSE 'No Change'
       END as trend
FROM ipo_master m
JOIN ipo_performance p ON m.ipo_id = p.ipo_id
ORDER BY change_since_listing DESC
'''

result5 = pd.read_sql(query5, conn)
print(result5)

                    company_name                  sector  listing_gain_pct  \
0                 Bharti Hexacom       Telecommunication             42.68   
1            Diffusion Engineers         Metals & Mining             20.93   
2                Waaree Energies                   Power             55.62   
3         Aadhar Housing Finance      Financial Services              4.59   
4                 Sagility India  Information Technology             -2.27   
5            ACME Solar Holdings                   Power            -12.40   
6     Niva Bupa Health Insurance      Financial Services              0.00   
7               Vishal Mega Mart                Services             43.50   
8             Sambhv Steel Tubes         Metals & Mining             19.01   
9            Hyundai Motor India              Automobile             -7.16   
10             Vodafone Idea FPO       Telecommunication             26.36   
11    Go Digit General Insurance      Financial Services        

In [12]:
query7 = '''
WITH ipo_data AS (
    SELECT m.ipo_id, m.company_name, m.sector, p.current_gain_pct
    FROM ipo_master m
    JOIN ipo_performance p ON m.ipo_id = p.ipo_id
)
SELECT a.company_name as company_a,
       b.company_name as company_b,
       a.sector,
       a.current_gain_pct as gain_a,
       b.current_gain_pct as gain_b,
       ROUND(a.current_gain_pct - b.current_gain_pct, 2) as gain_diff
FROM ipo_data a
JOIN ipo_data b
ON a.sector = b.sector AND a.ipo_id < b.ipo_id
ORDER BY a.sector, gain_diff DESC
'''

result7 = pd.read_sql(query7, conn)
print(result7)

                       company_a                     company_b  \
0            Hyundai Motor India                  Ola Electric   
1            Hyundai Motor India                 Kross Limited   
2                   Ola Electric                 Kross Limited   
3                      JNK India           Arisinfra Solutions   
4                      JNK India          Globe Civil Projects   
5                  Ceigall India           Arisinfra Solutions   
6                  Ceigall India          Globe Civil Projects   
7                      JNK India                 Ceigall India   
8          Afcons Infrastructure           Arisinfra Solutions   
9          Afcons Infrastructure          Globe Civil Projects   
10          Globe Civil Projects           Arisinfra Solutions   
11         Afcons Infrastructure                 Ceigall India   
12         Afcons Infrastructure                     JNK India   
13        Aadhar Housing Finance        HDB Financial Services   
14        

In [13]:
!pip install transformers torch --quiet
from transformers import pipeline

sentiment_analyzer = pipeline(
    "sentiment-analysis",
    model="ProsusAI/finbert"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [14]:
test_text = "The company shows strong revenue growth and low debt, but faces regulatory risks in international markets."
result = sentiment_analyzer(test_text)
print(result)


[{'label': 'positive', 'score': 0.5659626722335815}]


In [15]:
ipo_strengths = {
    1: "Bharti Hexacom has a strong customer base of over 27 million and consistent revenue growth at a 19.6% CAGR, backed by parent Bharti Airtel's resources and brand.",
    2: "Swiggy is a leading Indian food delivery and quick-commerce platform with strong revenue growth of 34% year-on-year and a diversified, tech-driven business spanning food delivery, Instamart, and Dineout.",
    3: "Hyundai Motor India benefits from an established global brand, a wide domestic dealership network, and a strong position in India's passenger vehicle market.",
    4: "NTPC Green Energy benefits from strong parent support from NTPC, the highest domestic credit ratings, and alignment with India's push toward 60 GW of renewable capacity by 2032.",
    5: "Vishal Mega Mart operates a large hypermarket network of 645 stores with a strong own-brand portfolio and revenue growth of over 17% in FY24.",
    6: "Sagility India provides established technology-enabled services to US healthcare payers and providers with a strong existing client base.",
    7: "Waaree Energies is India's largest solar PV module manufacturer with rapid revenue growth of nearly six-fold since FY21 and an expanding international presence including a US manufacturing facility.",
    8: "Premier Energies has 29 years of experience manufacturing solar PV cells and modules with growing manufacturing capacity in Hyderabad.",
    9: "Western Carriers is a multimodal logistics company with a diversified client base across steel, FMCG, and chemicals sectors.",
    10: "Diffusion Engineers is an established manufacturer of welding consumables with a strong foothold in a niche engineering product category.",
    11: "ACME Solar Holdings is a vertically integrated renewable energy company with in-house EPC and operations capabilities and a sizeable under-construction project pipeline.",
    12: "Niva Bupa Health Insurance is the third-largest standalone health insurer in India, backed by the Bupa Group, with strong premium growth and a healthy solvency ratio.",
    13: "Afcons Infrastructure is a large, diversified engineering and construction company with experience across multiple infrastructure segments and a substantial project pipeline.",
    14: "Brainbees Solutions (FirstCry) is India's largest multi-channel retail platform for mother, baby, and kids' products with a wide brand portfolio and growing international presence.",
    15: "Awfis Space Solutions is a leading flexible workspace operator in India with a strong footprint of leased centers across major cities.",
    16: "Ola Electric Mobility is India's largest electric two-wheeler manufacturer benefiting from favorable industry trends and strong government incentives for EV adoption.",
    17: "Bansal Wire Industries is a leading Indian steel wire manufacturer with a diversified customer base of over 5,000 clients across more than 50 countries.",
    18: "Kross Limited manufactures trailer axles and suspension assemblies with strong revenue growth of 44% CAGR from FY22 to FY24 and vertically integrated operations.",
    19: "BLS E-Services provides business correspondent banking and e-governance services across India, benefiting strongly from the government's Digital India push.",
    20: "Indegene is a digital-first life sciences commercialization company with an established global client base of pharmaceutical and biotech companies.",
    21: "Aadhar Housing Finance has an established presence since 2010 in affordable housing finance with a reasonably valued IPO relative to peers.",
    22: "Go Digit General Insurance is a digital-first, full-stack general insurer that has become the largest digital-first insurer in India with strong asset and revenue growth.",
    23: "JNK India has installations in nearly half of India's operating oil refineries and a decade of strong domestic and international project experience.",
    24: "Vodafone Idea is a major Indian telecom operator that successfully raised capital through its FPO to fund debt reduction and network expansion.",
    25: "Ceigall India is a fast-growing EPC contractor with a strong order book over 3 times FY24 revenue and consistent revenue, EBITDA, and profit growth of 50-67% CAGR.",
    26: "HDB Financial Services, backed by HDFC Bank, is the seventh-largest diversified retail-focused NBFC in India with a large, diversified loan book.",
    27: "Ellenbarrie Industrial Gases is one of India's oldest and largest wholly Indian-owned industrial gas manufacturers serving diverse strategic sectors.",
    28: "Globe Civil Projects is a New Delhi-based integrated EPC company with over 37 completed projects across 11 Indian states and a reasonable debt-equity ratio.",
    29: "Sambhv Steel Tubes is an established manufacturer of steel pipes and tubes serving the construction and infrastructure sectors.",
    30: "Arisinfra Solutions operates a strong B2B technology platform digitizing procurement of construction materials in a large, high-growth, under-digitized market."
}

ipo_risks = {
    1: "Bharti Hexacom's earnings have been volatile due to one-off gains, and it remains regionally concentrated with exposure to high industry churn and regulatory pricing pressure on ARPU.",
    2: "Swiggy has sustained net losses and negative operating cash flows since inception, faces intense competition, and carries pending legal proceedings worth over Rs 396 crore.",
    3: "Hyundai Motor India's IPO was structured entirely as an offer-for-sale with no fresh capital raised, and its shares declined post-listing, raising valuation and demand concerns.",
    4: "NTPC Green Energy carries risks typical of capital-intensive renewable infrastructure, including project execution timelines and dependence on continued government policy support.",
    5: "Vishal Mega Mart's IPO was entirely an offer-for-sale with no fresh capital, and it faces reliance on third-party vendors and intense competition from larger retailers.",
    6: "Sagility India's IPO was entirely an offer-for-sale with no fresh capital raised, and it carries revenue concentration risk in the US healthcare market.",
    7: "Waaree Energies relies on a small number of large customers and faces international market uncertainty through its export exposure.",
    8: "Premier Energies faces customer concentration, geographic concentration of manufacturing, and dependence on imported machinery from China.",
    9: "Western Carriers operates with thin margins typical of logistics businesses and faces fuel cost exposure and dependence on government infrastructure policy.",
    10: "Diffusion Engineers has limited scale as a small-cap company with customer concentration in industrial sectors and raw material price fluctuation risk.",
    11: "ACME Solar Holdings has concentrated revenue from just three states, heavy reliance on its top 10 off-takers for nearly 90% of revenue, and a history of losses excluding exceptional items.",
    12: "Niva Bupa Health Insurance has historically generated lower profitability than peers despite its market share, and faces high capital requirements and intense competition.",
    13: "Afcons Infrastructure faces typical EPC risks including project execution delays, working capital intensity, and dependence on government spending cycles.",
    14: "Brainbees Solutions has incurred losses partly due to ESOP and amortization charges, resulting in a negative P/E at IPO, and faces intense e-commerce competition.",
    15: "Awfis Space Solutions was loss-making at the EBIT level until 2023 and has high sector and city concentration risk along with long-term fixed lease costs.",
    16: "Ola Electric Mobility has reported sustained and widening losses exceeding Rs 1,584 crore in FY24, negative cash flows since inception, and high dependence on a single product line.",
    17: "Bansal Wire Industries operates in a highly competitive, fragmented industry with exposure to steel raw material price fluctuations and an aggressively priced issue.",
    18: "Kross Limited relies heavily on its top five customers for over 68% of revenue and carries valuation concerns given its premium pricing versus listed peers.",
    19: "BLS E-Services depends heavily on its promoter for e-governance project allocation and has revenue concentration with about 60% from a single large PSU bank.",
    20: "Indegene saw a sharp share price decline on listing day, reflecting risks from revenue concentration in life sciences clients and dependence on pharma outsourcing trends.",
    21: "Aadhar Housing Finance faces asset quality sensitivity in the affordable housing loan segment, interest rate exposure, and competition from banks and NBFCs.",
    22: "Go Digit General Insurance has a history of losses with a very high P/E reflecting weak past earnings, and faces underwriting risk in a highly competitive market.",
    23: "JNK India faces customer and sector concentration risk in the oil refining industry and revenue lumpiness typical of large, project-based equipment orders.",
    24: "Vodafone Idea carries very high existing debt levels, continued subscriber losses to rivals, and dependence on government relief measures for survival.",
    25: "Ceigall India relies heavily on government contracts and NHAI policy, has had negative operating and investing cash flows, and faces high working capital requirements.",
    26: "HDB Financial Services faces macroeconomic slowdown sensitivity, loan default risk in its uncollateralized unsecured lending portfolio, and heavy dependence on its parent.",
    27: "Ellenbarrie Industrial Gases faces customer concentration in industrial clients and competition from larger global gas producers in a capital-intensive business.",
    28: "Globe Civil Projects has high revenue concentration from a limited number of government clients, a declining project bid success rate, and dependence on third-party suppliers.",
    29: "Sambhv Steel Tubes faces steel price volatility and competitive intensity typical of the pipes and tubes manufacturing segment.",
    30: "Arisinfra Solutions posted losses through FY24 with only a recent turnaround, and faces customer concentration risk along with high working capital needs."
}

In [16]:
def to_signed(result):
    if result['label'] == 'positive':
        return result['score']
    elif result['label'] == 'negative':
        return -result['score']
    return 0

net_results = []
for ipo_id in ipo_strengths.keys():
    strength_result = sentiment_analyzer(ipo_strengths[ipo_id])[0]
    risk_result = sentiment_analyzer(ipo_risks[ipo_id])[0]

    strength_signed = to_signed(strength_result)
    risk_signed = to_signed(risk_result)
    net_score = round(strength_signed + risk_signed, 4)

    net_results.append({
        'ipo_id': ipo_id,
        'strength_signed': round(strength_signed, 4),
        'risk_signed': round(risk_signed, 4),
        'net_sentiment_score': net_score,
        'net_label': 'positive' if net_score > 0 else ('negative' if net_score < 0 else 'neutral')
    })

net_sentiment_df = pd.DataFrame(net_results)
print(net_sentiment_df)

    ipo_id  strength_signed  risk_signed  net_sentiment_score net_label
0        1           0.9436      -0.9555              -0.0119  negative
1        2           0.8680      -0.9686              -0.1007  negative
2        3           0.9037      -0.9686              -0.0648  negative
3        4           0.9188      -0.8610               0.0579  positive
4        5           0.8675      -0.9140              -0.0465  negative
5        6           0.8347      -0.7117               0.1231  positive
6        7           0.9192      -0.7867               0.1325  positive
7        8           0.8029      -0.8970              -0.0941  negative
8        9           0.0000      -0.9227              -0.9227  negative
9       10           0.8311       0.0000               0.8311  positive
10      11           0.0000      -0.9563              -0.9563  negative
11      12           0.9352      -0.9685              -0.0333  negative
12      13           0.0000      -0.9569              -0.9569  n

In [17]:
net_sentiment_df.to_sql('ipo_net_sentiment', conn, if_exists='replace', index=False)
conn.commit()
print("Net sentiment data loaded into SQL!")

Net sentiment data loaded into SQL!


In [20]:
query8_v2 = '''
SELECT m.company_name, m.sector,
       n.net_label, n.net_sentiment_score,
       p.current_gain_pct,
       CASE
           WHEN n.net_label = 'positive' AND p.current_gain_pct > 0 THEN 'Sentiment Matched'
           WHEN n.net_label = 'negative' AND p.current_gain_pct < 0 THEN 'Sentiment Matched'
           ELSE 'Sentiment Mismatched'
       END as accuracy_check
FROM ipo_master m
JOIN ipo_performance p ON m.ipo_id = p.ipo_id
JOIN ipo_net_sentiment n ON m.ipo_id = n.ipo_id
ORDER BY n.net_sentiment_score DESC
'''

result8_v2 = pd.read_sql(query8_v2, conn)
print(result8_v2)

                    company_name                  sector net_label  \
0                 BLS E-Services  Information Technology  positive   
1            Diffusion Engineers         Metals & Mining  positive   
2         Aadhar Housing Finance      Financial Services  positive   
3          Awfis Space Solutions                Services  positive   
4                Waaree Energies                   Power  positive   
5                 Sagility India  Information Technology  positive   
6              NTPC Green Energy                   Power  positive   
7         Bansal Wire Industries         Metals & Mining   neutral   
8                 Bharti Hexacom       Telecommunication  negative   
9              Vodafone Idea FPO       Telecommunication  negative   
10                  Ola Electric              Automobile  negative   
11    Go Digit General Insurance      Financial Services  negative   
12    Niva Bupa Health Insurance      Financial Services  negative   
13                 C

In [19]:
query9 = '''
WITH accuracy_calc_v2 AS (
    SELECT m.ipo_id,
    CASE
        WHEN n.net_label = 'positive' AND p.current_gain_pct > 0 THEN 'Sentiment Matched'
        WHEN n.net_label = 'negative' AND p.current_gain_pct < 0 THEN 'Sentiment Matched'
        ELSE 'Sentiment Mismatched'
    END as accuracy_check
    FROM ipo_master m
    JOIN ipo_performance p ON m.ipo_id = p.ipo_id
    JOIN ipo_net_sentiment n ON m.ipo_id = n.ipo_id
)
SELECT
    accuracy_check,
    COUNT(*) as count,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM accuracy_calc_v2), 2) as pct_of_total
FROM accuracy_calc_v2
GROUP BY accuracy_check
'''

result9 = pd.read_sql(query9, conn)
print(result9)

         accuracy_check  count  pct_of_total
0     Sentiment Matched     17         56.67
1  Sentiment Mismatched     13         43.33


In [21]:
query10_v2 = '''
SELECT m.company_name, n.net_sentiment_score, p.current_gain_pct,
       NTILE(3) OVER (ORDER BY n.net_sentiment_score) as score_tercile
FROM ipo_master m
JOIN ipo_performance p ON m.ipo_id = p.ipo_id
JOIN ipo_net_sentiment n ON m.ipo_id = n.ipo_id
'''

result10_v2 = pd.read_sql(query10_v2, conn)
print(result10_v2.groupby('score_tercile')['current_gain_pct'].mean())

score_tercile
1   -17.648
2     8.736
3    52.192
Name: current_gain_pct, dtype: float64
